In [17]:
import pandas as pd
df_aime = pd.read_parquet("/scratch/s6019595/llm-think-too-much/experiments/L1-Regressor/aime-250_results.parquet")

#Rename generated_think_tokens to token_count
df_aime = df_aime.rename(columns={"generated_text": "generated"})
df_aime.to_parquet("/scratch/s6019595/llm-think-too-much/experiments/L1-Regressor/aime-250_results.parquet", index=False)
df_math = pd.read_parquet("/scratch/s6019595/llm-think-too-much/experiments/L1-Regressor/math-500_results.parquet")
df_math = df_math.rename(columns={"generated_text": "generated"})
df_math.to_parquet("/scratch/s6019595/llm-think-too-much/experiments/L1-Regressor/math-500_results.parquet", index=False)

In [14]:
from pathlib import Path
import pandas as pd

results_path = Path().resolve().parent / "experiments"
models = [
    "Qwen3-8B_nowait",
    "Qwen3-8B_baseline",
    "L1-Regressor"
]

datasets = ["math-500", "amc", "aime-250", "gsm8k", "olympiad"] 

rows = []
for model in models:
    model_dir = results_path / model
    for dataset in datasets:
        parquet_file = model_dir / f"{dataset}_results.parquet"
        if not parquet_file.exists():
            continue
        df = pd.read_parquet(parquet_file)
        total = len(df)
        correct = df["is_correct"].sum()
        accuracy = correct / total if total > 0 else 0.0
        avg_tokens = df["token_count"].mean()
        rows.append({
            "model": model,
            "dataset": dataset,
            "accuracy": accuracy,
            "num_correct": correct,
            "num_total": total,
            "avg_tokens": avg_tokens,
        })

summary = pd.DataFrame(rows)
summary

,model,dataset,accuracy,num_correct,num_total,avg_tokens
0,Qwen3-8B_nowait,math-500,0.889780,444,499,3962.032064
1,Qwen3-8B_nowait,amc,0.950000,38,40,6506.375000
2,Qwen3-8B_nowait,aime-250,0.756000,189,250,10270.148000
3,Qwen3-8B_nowait,gsm8k,0.944655,1246,1319,1184.338135
4,Qwen3-8B_nowait,olympiad,0.629080,424,674,8580.488131
5,Qwen3-8B_baseline,math-500,0.895792,447,499,5491.406814
6,Qwen3-8B_baseline,amc,0.950000,38,40,8699.475000
7,Qwen3-8B_baseline,aime-250,0.792000,198,250,14156.416000
8,Qwen3-8B_baseline,gsm8k,0.940864,1241,1319,1906.765732
9,Qwen3-8B_baseline,olympiad,0.646884,436,674,12015.155786


In [39]:
from pathlib import Path
import pandas as pd
from eval_pipeline import is_equiv
model_qwen = "Qwen3-8B_baseline"
model_qwen_nowait = "Qwen3-8B_nowait"
model_l1 = "L1-Regressor"

df_qwen_math = pd.read_parquet(Path().resolve().parent / "experiments" / model_l1 / "aime-250_results.parquet")
print("shape", df_qwen_math.shape)
print("Accuracy", df_qwen_math["is_correct"].mean())
incorrect = df_qwen_math[df_qwen_math["is_correct"] == False].copy()
print("Number of incorrect answers:", len(incorrect))
df_qwen_math.head()

shape (250, 9)
Accuracy 1.0
Number of incorrect answers: 0


,question_id,prompt,solution,generated_think_text,generated,target_think_tokens,token_count,latency_sec,is_correct
0,aime_10,<|im_start|>user\nThe solid shown has a square...,288,"Okay, let's try to figure out the volume of th...",<|im_start|>user\nThe solid shown has a square...,2678,1461,1.896769,True
1,aime_107,<|im_start|>user\nFor how many real numbers $a...,10,"Okay, let's see. The problem is asking for how...",<|im_start|>user\nFor how many real numbers $a...,1389,776,1.896769,True
2,aime_109,"<|im_start|>user\nTwo three-letter strings, $a...",532,"Okay, let's try to figure out this probability...","<|im_start|>user\nTwo three-letter strings, $a...",4742,3478,1.896769,True
3,aime_110,<|im_start|>user\nTwelve congruent disks are p...,135,"Okay, so there's this problem with twelve cong...",<|im_start|>user\nTwelve congruent disks are p...,4226,3048,1.896769,True
4,aime_118,<|im_start|>user\nLet $S^{}_{}$ be the set of ...,660,"Okay, so I need to figure out how many differe...",<|im_start|>user\nLet $S^{}_{}$ be the set of ...,2678,1558,1.896769,True


In [36]:
"""
Normalization and equivalence checking for LaTeX math answers.

Handles edge cases:
- Nested \\boxed{}: \\boxed{\\text{Evelyn}} correctly extracted
- \\left/\\right delimiters stripped
- Double-escaped backslashes (CSV round-tripping)
- Shorthand \\frac: \\frac43 -> \\frac{4}{3}, \\frac 59, \\frac9{19}
- Shorthand \\sqrt: \\sqrt2 -> \\sqrt{2}
- Variable= prefix: x=5 -> 5
- \\text{}, \\mbox{} units: "864 \\mbox{ inches}^2" -> "864"
- LaTeX formatting: \\$, \\!, \\,, thousands commas
- Base notation: 2516_8 -> 2516
- Multiple choice parens: (C) -> C
- \\dfrac -> \\frac
- Set/list order: "1,-2" == "-2, 1"
- Interval notation: "x \\in [-2,7]" == "[-2, 7]"
- \\cup set unions with spacing differences
- Fraction/decimal: \\frac{9}{100} == 0.09
- Algebraic equivalence via sympy: \\frac{11+9a}{20} == \\frac{9a+11}{20}
"""

import re


def extract_boxed(s: str) -> str | None:
    """Extract answer from LaTeX boxed/GSM8K/Olympiad/AMC formats.

    Handles nested braces (e.g. \\boxed{\\text{Evelyn}}) by parsing brace depth.
    Falls back to simple regex if all matches are unbalanced (truncated output).
    """
    if not s:
        return None

    # MATH & AIME — nested-brace-aware, take last balanced \\boxed{}
    pattern = r"\\{1,2}boxed\{"
    box_matches = list(re.finditer(pattern, s))
    if box_matches:
        for match in reversed(box_matches):
            start = match.end()
            depth = 1
            i = start
            while i < len(s) and depth > 0:
                if s[i] == "{":
                    depth += 1
                elif s[i] == "}":
                    depth -= 1
                i += 1

            if depth != 0:
                continue  # Unbalanced — try previous match

            content = s[start : i - 1].strip()

            # Unwrap \text{...}, \textbf{...}, etc.
            text_match = re.match(r"\\text(?:bf|it|rm|sf)?\{(.+)\}$", content)
            if text_match:
                content = text_match.group(1).strip()

            return content

        # All unbalanced — fall back to simple regex
        simple = re.findall(r"\\{1,2}boxed\{([^}]*)\}", s)
        if simple:
            return simple[-1].strip()

    # GSM8K: #### <answer>
    matches = re.findall(r"(?m)^[ \t]*####[ \t]*([^\n\r#]+?)[ \t]*$", s)
    if matches:
        return matches[-1].strip()

    # Olympiad: last $...$
    matches = re.findall(r"\$([^$]*)\$", s)
    if matches:
        return matches[-1].strip()

    # AMC: last standalone number
    matches = re.findall(r"(?m)^[ \t]*([+-]?\d+(?:\.\d+)?)[ \t]*$", s)
    if matches:
        return matches[-1].strip()

    return s


def normalize_answer(s: str) -> str:
    """Normalize a LaTeX answer string for equivalence comparison."""
    if not s or s == "nan":
        return s

    # Fix double-escaped backslashes (e.g. from CSV round-tripping)
    while "\\\\" in s:
        s = s.replace("\\\\", "\\")

    # Strip \left / \right delimiters
    s = s.replace("\\left(", "(").replace("\\right)", ")")
    s = s.replace("\\left[", "[").replace("\\right]", "]")
    s = s.replace("\\left", "").replace("\\right", "")

    # Strip \text{}, \mbox{} with optional trailing exponent (e.g. ^2)
    s = re.sub(
        r"\s*\\(?:text|mbox|textbf|mathrm)\{[^}]*\}(?:\^\d+)?\s*$", "", s
    ).strip()
    s = re.sub(
        r"\s*\\(?:text|mbox|textbf|mathrm)\{[^}]*\}(?:\^\d+)?", "", s
    ).strip()

    # Strip \$ (LaTeX literal dollar sign)
    s = s.replace("\\$", "")

    # Strip \! (thin neg space) and \, (thin space)
    s = s.replace("\\!", "").replace("\\,", "")

    # Strip "x \in" prefix from intervals
    s = re.sub(r"^[a-zA-Z]\s*\\in\s*", "", s).strip()

    # Strip ^\circ (degree symbol)
    s = re.sub(r"\^\\circ\s*$", "", s).strip()

    # Strip base notation suffix: 2516_8 -> 2516, 4210_{5} -> 4210
    s = re.sub(r"_\{?\d+\}?\s*$", "", s).strip()

    # Strip variable= prefix: x=5 -> 5
    s = re.sub(r"^[a-zA-Z]\s*=\s*", "", s).strip()

    # Unwrap single-letter parens: (C) -> C
    m = re.match(r"^\(([A-Za-z])\)$", s)
    if m:
        s = m.group(1)

    # \dfrac -> \frac
    s = s.replace("\\dfrac", "\\frac")

    # Normalize shorthand \sqrt: \sqrt2 -> \sqrt{2} (single non-brace char)
    s = re.sub(r"\\sqrt([^{\s\\])", r"\\sqrt{\1}", s)

    # Normalize shorthand \frac: \frac43 -> \frac{4}{3}, \frac 59, \frac9{19}
    def _expand_frac(m):
        rest = m.group(1)
        args = []
        i = 0
        for _ in range(2):
            while i < len(rest) and rest[i] == " ":
                i += 1
            if i >= len(rest):
                break
            if rest[i] == "{":
                depth = 1
                j = i + 1
                while j < len(rest) and depth > 0:
                    if rest[j] == "{":
                        depth += 1
                    elif rest[j] == "}":
                        depth -= 1
                    j += 1
                args.append(rest[i:j])
                i = j
            else:
                args.append("{" + rest[i] + "}")
                i += 1
        if len(args) == 2:
            return "\\frac" + args[0] + args[1]
        return m.group(0)

    s = re.sub(r"\\frac(.*)", _expand_frac, s)

    # Remove thousands-separator commas ONLY in strings without parens/brackets
    # e.g. "58,500" -> "58500" but NOT "(2,12)" or "-2,1"
    if not any(c in s for c in "()[]\\"):
        s = re.sub(r"(?<=\d),(?=\d{3}(?:\D|$))", "", s)

    # Normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s


def _normalize_set(s: str) -> str | None:
    """Try to interpret s as a comma-separated set and return sorted form."""
    inner = s.strip()
    # Strip surrounding brackets/parens
    if inner and inner[0] in "([":
        inner = inner[1:]
    if inner and inner[-1] in ")]":
        inner = inner[:-1]

    parts = [p.strip() for p in inner.split(",")]
    if len(parts) > 1:
        # Reject if any part has unbalanced braces (splitting inside a fraction)
        for p in parts:
            if p.count("{") != p.count("}"):
                return None
        return ",".join(sorted(parts))
    return None


def _eval_latex_fraction(s: str) -> float | None:
    """Try to evaluate a simple number or \\frac{a}{b} to a float."""
    try:
        return float(s)
    except ValueError:
        pass
    m = re.match(r"^\\frac\{([^}]+)\}\{([^}]+)\}$", s)
    if m:
        try:
            return float(m.group(1)) / float(m.group(2))
        except (ValueError, ZeroDivisionError):
            pass
    return None


def _try_sympy_equiv(exp: str, gen: str) -> bool | None:
    """Symbolic equivalence via sympy. Returns None if parsing fails."""
    try:
        from sympy.parsing.latex import parse_latex
        from sympy import simplify

        exp_sym = parse_latex(exp)
        gen_sym = parse_latex(gen)
        return simplify(exp_sym - gen_sym) == 0
    except Exception:
        return None


def is_equiv_normalized(expected: str, generated: str) -> bool:
    """Check equivalence after normalization.

    Layers (in order):
    1. Exact match after normalization
    2. Exact match ignoring spaces
    3. Set/list comparison (order-independent)
    4. Numeric fraction/decimal comparison
    5. Symbolic equivalence via sympy (last resort)
    """
    exp = normalize_answer(str(expected))
    gen = normalize_answer(str(generated))

    # 1. Exact
    if exp == gen:
        return True

    # 2. Ignore spaces
    if exp.replace(" ", "") == gen.replace(" ", ""):
        return True

    # 3. Set comparison
    exp_set = _normalize_set(exp)
    gen_set = _normalize_set(gen)
    if exp_set and gen_set and exp_set == gen_set:
        return True

    # 4. Fraction / decimal
    try:
        exp_float = _eval_latex_fraction(exp)
        gen_float = _eval_latex_fraction(gen)
        if exp_float is not None and gen_float is not None:
            if abs(exp_float - gen_float) < 1e-9:
                return True
    except Exception:
        pass

    # 5. Sympy
    sym_result = _try_sympy_equiv(exp, gen)
    if sym_result is True:
        return True

    return False

def evaluate_answer(expected_answer: str, generated_answer: str) -> bool:
    exp_val = extract_boxed(expected_answer)
    gen_val = extract_boxed(generated_answer)
    if exp_val is None or gen_val is None:
        return False, exp_val, gen_val
    return is_equiv_normalized(gen_val, exp_val), exp_val, gen_val

In [40]:
df_qwen_math['expected_value'] = df_qwen_math['solution'].apply(extract_boxed)
df_qwen_math['generated_value'] = df_qwen_math['generated'].apply(extract_boxed)
df_qwen_math['is_correct'] = df_qwen_math.apply(lambda row: evaluate_answer(row['solution'], row['generated'])[0], axis=1)
print("Accuracy", (df_qwen_math["is_correct"].sum()) / len(df_qwen_math))

Accuracy 0.564


In [42]:
#all incorrect
incorrect = df_qwen_math[df_qwen_math["is_correct"] == False].copy()
print(f"Number of incorrect answers: {len(incorrect)}")
incorrect.head(50)

Number of incorrect answers: 109


,question_id,prompt,solution,generated_think_text,generated,target_think_tokens,token_count,latency_sec,is_correct,expected_value,generated_value
0,aime_10,<|im_start|>user\nThe solid shown has a square...,288,"Okay, let's try to figure out the volume of th...",<|im_start|>user\nThe solid shown has a square...,2678,1461,1.896769,False,288,432
2,aime_109,"<|im_start|>user\nTwo three-letter strings, $a...",532,"Okay, let's try to figure out this probability...","<|im_start|>user\nTwo three-letter strings, $a...",4742,3478,1.896769,False,532,266
3,aime_110,<|im_start|>user\nTwelve congruent disks are p...,135,"Okay, so there's this problem with twelve cong...",<|im_start|>user\nTwelve congruent disks are p...,4226,3048,1.896769,False,135,30
4,aime_118,<|im_start|>user\nLet $S^{}_{}$ be the set of ...,660,"Okay, so I need to figure out how many differe...",<|im_start|>user\nLet $S^{}_{}$ be the set of ...,2678,1558,1.896769,False,660,648
6,aime_136,<|im_start|>user\nTwo thousand points are give...,118,"Okay, let's try to figure out this problem. So...",<|im_start|>user\nTwo thousand points are give...,1647,929,1.896769,False,118,6
8,aime_139,<|im_start|>user\nThe vertices of $\triangle A...,344,"Okay, so we need to find k + m where P1 = (k, ...",<|im_start|>user\nThe vertices of $\triangle A...,873,185,1.896769,False,344,212
9,aime_141,<|im_start|>user\nA rectangle that is inscribe...,448,"Okay, so I need to find the smallest perimeter...",<|im_start|>user\nA rectangle that is inscribe...,4742,3415,1.896769,False,448,392
10,aime_155,<|im_start|>user\nCircles of radius $3$ and $6...,224,"Okay, let's try to solve this geometry problem...",<|im_start|>user\nCircles of radius $3$ and $6...,3710,2395,1.896769,False,224,144
11,aime_158,<|im_start|>user\nFor how many ordered pairs o...,85,"Okay, so I need to find the number of ordered ...",<|im_start|>user\nFor how many ordered pairs o...,2421,1302,1.896769,False,85,27
13,aime_168,"<|im_start|>user\nA wooden cube, whose edges a...",166,"Okay, let's try to figure out this cube shadow...","<|im_start|>user\nA wooden cube, whose edges a...",5000,3851,1.896769,False,166,253


In [ ]:

false_negatives_math = ["13081-math-500","13093-math-500","12991-math-500", "12772-math-500"]
false_negatives_aime =  ["15439-aime-250"]
false_negatives_gsm8k = ["13266-gsm8k","14174-gsm8k", "13979-gsm8k"]
false_negatives_amc = []
false_negatives_olympiad = [] 

print("expected_value:", incorrect[incorrect["unique_id"] == "13081-math-500"]["expected_value"].iloc[0])
print("generated_value:", incorrect[incorrect["unique_id"] == "13081-math-500"]["generated_value"].iloc[0])
print("-"*60)
print("expected_value:", incorrect[incorrect["unique_id"] == "13093-math-500"]["expected_value"].iloc[0])
print("generated_value:", incorrect[incorrect["unique_id"] == "13093-math-500"]["generated_value"].iloc[0])
print("-"*60)
print("expected_value:", incorrect[incorrect["unique_id"] == "12991-math-500"]["expected_value"].iloc[0])
print("generated_value:", incorrect[incorrect["unique_id"] == "12991-math-500"]["generated_value"].iloc[0])
print("-"*60)
print("expected_value:", incorrect[incorrect["unique_id"] == "12772-math-500"]["expected_value"].iloc[0])
print("generated_value:", incorrect[incorrect["unique_id"] == "12772-math-500"]["generated_value"].iloc[0])